[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Digital-AI-Finance/Introduction-to-Machine-Learning-notebooks/blob/master/svm_basics.ipynb)

# Support vector machines, in pictures

Run each cell with **Shift and Enter**. Five tasks, each one number to change,
and the answer written underneath.

A support vector machine separates two groups with the line that leaves the
widest empty street on either side of it.

## 1. The widest street between two kinds

Two kinds of iris flower, measured by petal length and petal width in
centimetres. Many straight lines separate them, and the machine takes the one
with the widest empty street around it.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def street(model, X, y, title, deleted=None):
    """The line, the street around it, and the points that hold it in place."""
    seen = X if deleted is None or not len(deleted) else np.vstack([X, deleted])
    gx, gy = np.meshgrid(np.linspace(seen[:, 0].min() - 0.5, seen[:, 0].max() + 0.5, 300),
                         np.linspace(seen[:, 1].min() - 0.5, seen[:, 1].max() + 0.5, 300))
    score = model.decision_function(np.c_[gx.ravel(), gy.ravel()]).reshape(gx.shape)

    fig, ax = plt.subplots(figsize=(5.6, 3.6))
    ax.contourf(gx, gy, score, levels=[-1e6, 0, 1e6], colors=["#dfe4ec", "#fbeddc"])
    ax.contour(gx, gy, score, levels=[-1, 0, 1], colors="#b45309",
               linestyles=["--", "-", "--"], linewidths=1.2)
    for k in (0, 1):
        ax.scatter(X[y == k, 0], X[y == k, 1], marker="os"[k], s=18,
                   color=["#1e3a5f", "#b45309"][k], label=names[k], zorder=3)
    if deleted is not None and len(deleted):
        ax.scatter(deleted[:, 0], deleted[:, 1], marker="x", s=22,
                   color="#64748b", label="deleted", zorder=3)
    ax.scatter(model.support_vectors_[:, 0], model.support_vectors_[:, 1], s=130,
               facecolors="none", edgecolors="#1e3a5f", linewidths=1.4, zorder=4)
    ax.set_xlabel(labels[0])
    ax.set_ylabel(labels[1])
    ax.set_title(title)
    ax.legend(frameon=False, fontsize=8, loc="upper left")
    plt.show()

from sklearn.datasets import load_iris
from sklearn.svm import SVC

iris = load_iris()
two = iris.target < 2
X = iris.data[two][:, [2, 3]]
y = iris.target[two]
labels = ["petal length (cm)", "petal width (cm)"]
names = ["setosa", "versicolor"]

drop = 0
everyone = SVC(kernel="linear", C=1).fit(X, y)
off_the_kerb = np.setdiff1d(np.arange(len(X)), everyone.support_)
gone = off_the_kerb[:drop]
keep = np.setdiff1d(np.arange(len(X)), gone)
machine = SVC(kernel="linear", C=1).fit(X[keep], y[keep])

print("flowers used:", len(keep))
print("flowers on the kerb:", X[keep][machine.support_].tolist())
print("width of the street: %.3f" % (2 / np.linalg.norm(machine.coef_[0])))

street(machine, X[keep], y[keep], "the widest street", X[gone])

The solid line is the boundary and the dashed lines are the kerbs of the
street around it. The ringed flowers stand on a kerb, and where the line runs
depends on those flowers alone.

**Task 1.** Change `drop = 0` to `drop = 90` and run the cell again. That
deletes 90 flowers that are off the kerb before the machine is fitted.

*Answer.* Ninety flowers vanish, shown as grey crosses, and the line, the street and the
two ringed flowers stay where they were. The street is still 1.534 wide. Only
the flowers on the kerb decide the line.

## 2. What a point inside the street costs

These two kinds overlap, so no street can be empty. The machine lets some
points stand inside it or across the line, and `C` is the price of each. A
small `C` makes that cheap and a large `C` makes it dear.

In [ ]:
from sklearn.model_selection import train_test_split

two = iris.target > 0
X = iris.data[two][:, [2, 3]]
y = iris.target[two] - 1
names = ["versicolor", "virginica"]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.4, random_state=0)

C = 1
machine = SVC(kernel="linear", C=C).fit(Xtr, ytr)

print("points on the kerb or inside the street:", len(machine.support_))
print("width of the street: %.3f" % (2 / np.linalg.norm(machine.coef_[0])))
print("right on the rows it learned from: %.3f" % machine.score(Xtr, ytr))
print("right on the rows held back: %.3f" % machine.score(Xte, yte))

street(machine, Xtr, ytr, "C = %g" % C)

**Task 2.** Change `C = 1` to `C = 100` and run the cell again.

*Answer.* The street narrows from 0.825 wide to 0.285, and the points on the kerb or
inside it fall from 12 to 5. The score on the rows it learned from goes from
0.967 to 0.983 and the score on the rows held back goes from 0.900 to 0.875: a
dearer crossing narrows the street until it fits the rows it learned from a
little better, and the rows held back score a little worse. Try `C = 0.01` as
well. The street swells to 5.206 wide, 54 of the 60 points sit inside it, and
the score on the rows held back falls to 0.475.

## 3. The units decide

A machine that measures distance reads every column in the units it was given.
Below are 300 customers with two columns, age in years and income in francs.
The two groups differ in age and have the same income.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def zones(ax, model, X, y, title):
    """The boundary a fitted model draws, with the points it was given."""
    gx, gy = np.meshgrid(np.linspace(X[:, 0].min() - 0.3, X[:, 0].max() + 0.3, 250),
                         np.linspace(X[:, 1].min() - 0.3, X[:, 1].max() + 0.3, 250))
    zone = model.predict(np.c_[gx.ravel(), gy.ravel()]).reshape(gx.shape)
    ax.contourf(gx, gy, zone, levels=[-0.5, 0.5, 1.5], colors=["#dfe4ec", "#fbeddc"])
    ax.contour(gx, gy, zone, levels=[0.5], colors="#b45309", linewidths=1.0)
    for k in (0, 1):
        ax.scatter(X[y == k, 0], X[y == k, 1], marker="os"[k], s=14,
                   color=["#1e3a5f", "#b45309"][k], label=names[k], zorder=3)
    ax.set_xlabel(labels[0])
    ax.set_ylabel(labels[1])
    ax.set_title(title)

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(0)
group = rng.integers(0, 2, 300)
age = np.where(group == 0, rng.normal(32, 5, 300), rng.normal(48, 5, 300))
income = rng.normal(80000, 8000, 300)

unit = 1
X = np.c_[age, income / unit]
Xtr, Xte, ytr, yte = train_test_split(X, group, test_size=0.4, random_state=0)
labels = ["age (years)", "income (francs / %d)" % unit]
names = ["group 1", "group 2"]

raw = SVC().fit(Xtr, ytr)
scaled = make_pipeline(StandardScaler(), SVC()).fit(Xtr, ytr)

print("income divided by %d" % unit)
print("right on the rows held back, columns as they are: %.3f"
      % raw.score(Xte, yte))
print("right on the rows held back, each column scaled: %.3f"
      % scaled.score(Xte, yte))

fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.6))
zones(axes[0], raw, Xtr, ytr, "columns as they are")
zones(axes[1], scaled, Xtr, ytr, "each column scaled")
plt.tight_layout()
plt.show()

The left machine is fitted on the columns as they come. The right one is fitted
after each column is rescaled to the same spread.

**Task 3.** Change `unit = 1` to `unit = 1000` and run the cell again. Income is
then divided by 1000, which counts it in thousands of francs.

*Answer.* The score on the left goes from 0.533 to 0.958, which is what the machine on
the right scored all along, and the boundary on the left turns into a nearly
vertical cut at an age of about 40. With income counted in francs, a difference
of one thousand francs outweighs a difference of ten years, so the machine on
the left answers the same way whatever the age. Counted in thousands, the two
columns weigh about the same and age can be heard. Rescaling every column to the
same spread, as the machine on the right did, gives that for any table.

## 4. A boundary that curves

Here one group sits in a ring around the other, and no straight line separates
them. The kernel is the machine's way of drawing a boundary: `"linear"` draws a
line, and `"rbf"` lets the boundary curve around whichever points lie close
together.

In [ ]:
from sklearn.datasets import make_circles

X, y = make_circles(n_samples=400, noise=0.1, factor=0.4, random_state=0)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.4, random_state=0)
labels = ["first measurement", "second measurement"]
names = ["outer ring", "inner disc"]

kernel = "linear"
machine = SVC(kernel=kernel).fit(Xtr, ytr)

print("kernel:", kernel)
print("right on the rows it learned from: %.3f" % machine.score(Xtr, ytr))
print("right on the rows held back: %.3f" % machine.score(Xte, yte))

fig, ax = plt.subplots(figsize=(4.8, 3.6))
zones(ax, machine, Xtr, ytr, 'kernel "%s"' % kernel)
plt.show()

**Task 4.** Change `kernel = "linear"` to `kernel = "rbf"` and run the cell
again.

*Answer.* The boundary closes into a circle around the inner disc, and the score on the
rows held back goes from 0.600 to 1.000. A straight line has to leave part of
the ring on the same side as the disc, which is why it stopped at 0.600.

## 5. How local the curve is

The `"rbf"` kernel has one more number, `gamma`, which sets how far the reach of
a single point extends. A small `gamma` lets each point count over a wide
neighbourhood and a large `gamma` confines it to the points beside it.

In [ ]:
from sklearn.datasets import make_moons

X, y = make_moons(n_samples=400, noise=0.28, random_state=0)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.4, random_state=0)
labels = ["first measurement", "second measurement"]
names = ["group 1", "group 2"]

gamma = 1
machine = SVC(kernel="rbf", gamma=gamma).fit(Xtr, ytr)

print("points holding the boundary:", len(machine.support_))
print("right on the rows it learned from: %.3f" % machine.score(Xtr, ytr))
print("right on the rows held back: %.3f" % machine.score(Xte, yte))

fig, ax = plt.subplots(figsize=(5.0, 3.6))
zones(ax, machine, Xtr, ytr, "gamma = %g" % gamma)
plt.show()

**Task 5.** Change `gamma = 1` to `gamma = 100` and run the cell again.

*Answer.* The smooth S shaped boundary breaks into small islands around single points.
The points holding it up go from 77 to 219 of 240, the score on the rows it
learned from rises from 0.917 to 0.967, and the score on the rows held back
falls from 0.919 to 0.875. A large `gamma` lets the machine memorize its rows.
Try `gamma = 0.01` as well: the boundary loosens and both scores fall, to 0.821
on the rows it learned from and 0.887 on the rows held back.

## What you can say now

A support vector machine separates two groups with the widest empty street it
can find, and only the points on the kerb decide where that street runs. `C`
sets the price of a point inside the street, the kernel sets whether the
boundary may curve, and `gamma` sets how local the curve is. The machine
measures distance, so every column has to be counted in comparable units.